In [ ]:
# Setup: import core analytics libs and list Kaggle input files.
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
# -*- coding: utf-8 -*-
"""01-quickstart-llamatelemetry-v0-1-0-e3 (Corrected for Kaggle)"""

# Base imports and typing helpers used across the notebook.

import os
import subprocess
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, Any, Optional, List


In [2]:
# Check GPU availability and CUDA/VRAM details in the Kaggle runtime.

print("=" * 72)
print("KAGGLE GPU ENVIRONMENT CHECK")
print("=" * 72)

gpu_result = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
gpu_lines = [line for line in gpu_result.stdout.splitlines() if line.startswith("GPU")]

print(f"Detected GPUs: {len(gpu_lines)}")
for line in gpu_lines:
    print(" ", line)

print("\nCUDA version:")
print(subprocess.run(["nvcc", "--version"], capture_output=True, text=True).stdout)

print("\nVRAM summary:")
print(subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)

if len(gpu_lines) < 2:
    print("\nWarning: This notebook expects Kaggle GPU T4 x2.")
else:
    print("\nReady: dual-GPU Kaggle environment detected.")

KAGGLE GPU ENVIRONMENT CHECK
Detected GPUs: 2
  GPU 0: Tesla T4 (UUID: GPU-54b45a17-4a47-26b8-4dd3-9b53c4da4076)
  GPU 1: Tesla T4 (UUID: GPU-5e680704-e39b-2ab8-d949-137f5f94bb1f)

CUDA version:
nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Thu_Jun__6_02:18:23_PDT_2024
Cuda compilation tools, release 12.5, V12.5.82
Build cuda_12.5.r12.5/compiler.34385749_0


VRAM summary:
index, name, memory.total [MiB]
0, Tesla T4, 15360 MiB
1, Tesla T4, 15360 MiB


Ready: dual-GPU Kaggle environment detected.


In [3]:
# Install llamatelemetry from GitHub and print the installed version.


!pip install -q --no-cache-dir --force-reinstall git+https://github.com/llamatelemetry/llamatelemetry.git@v0.1.1

import llamatelemetry
print("\nllamatelemetry version:", llamatelemetry.__version__)

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 612.9/612.9 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 308.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 290.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 331.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 347.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 292.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 246.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.7/153.7 kB 341.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.3/196.3 kB 381.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.5/20


🎯 llamatelemetry v0.1.1 First-Time Setup - Kaggle 2× T4 Multi-GPU

🎮 GPU Detected: Tesla T4 (Compute 7.5)
  ✅ Tesla T4 detected - Perfect for llamatelemetry v0.1.1!
🌐 Platform: Kaggle

📦 Downloading Kaggle 2× T4 binaries (~961 MB)...
    Features: FlashAttention + Tensor Cores + Multi-GPU tensor-split

➡️  Attempt 1: HuggingFace (llamatelemetry-v0.1.1-cuda12-kaggle-t4x2.tar.gz)
📥 Downloading v0.1.1 from HuggingFace Hub...
   Repo: waqasm86/llamatelemetry-binaries
   File: v0.1.1/llamatelemetry-v0.1.1-cuda12-kaggle-t4x2.tar.gz


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


v0.1.1/llamatelemetry-v0.1.1-cuda12-kagg(…):   0%|          | 0.00/1.40G [00:00<?, ?B/s]

v0.1.1/llamatelemetry-v0.1.1-cuda12-kagg(…):   0%|          | 0.00/130 [00:00<?, ?B/s]

🔐 Verifying SHA256 checksum...
   ✅ Checksum verified
📦 Extracting llamatelemetry-v0.1.1-cuda12-kaggle-t4x2.tar.gz...
Found 21 files in archive
Extracted 21 files to /root/.cache/llamatelemetry/extract_0.1.1
✅ Extraction complete!
  Found bin/ and lib/ under /root/.cache/llamatelemetry/extract_0.1.1/llamatelemetry-v0.1.1-cuda12-kaggle-t4x2
  Copied 13 binaries to /usr/local/lib/python3.12/dist-packages/llamatelemetry/binaries/cuda12
  Copied 2 libraries to /usr/local/lib/python3.12/dist-packages/llamatelemetry/lib
✅ Binaries installed successfully!


llamatelemetry version: 0.1.1


In [4]:
# Detect CUDA and print detailed GPU metadata.
import llamatelemetry as lt

# detect_cuda() probes the NVIDIA driver and returns GPU metadata
cuda_info = lt.detect_cuda()

print(f"CUDA available: {cuda_info['available']}")
print(f"CUDA version:   {cuda_info['version']}")

for gpu in cuda_info["gpus"]:
    print(f"  {gpu['name']} | {gpu['memory']} MB | "
          f"SM {gpu['compute_capability']} | "
          f"Driver {gpu['driver_version']}")

CUDA available: True
CUDA version:   12.5
  Tesla T4 | 15360 MiB MB | SM 7.5 | Driver 580.105.08
  Tesla T4 | 15360 MiB MB | SM 7.5 | Driver 580.105.08


In [5]:
# Initialize the inference engine with telemetry enabled.
engine = lt.InferenceEngine(
    server_url="http://127.0.0.1:8080",  # default server address
    enable_telemetry=True,               
)

In [6]:
# Load the model with explicit server/runtime configuration.
engine.unload_model()

engine.load_model(
    "gemma-3-1b-Q4_K_M",   # registry name
    auto_start=True,         # start llama-server after loading
    auto_configure=True,     # auto-set gpu_layers, ctx_size based on GPU
    gpu_layers=99,         # None = auto-detect from VRAM
    ctx_size=4096,           # None = auto-detect
    n_parallel=4,           # number of parallel request slots
    verbose=True,            # print loading progress
    report_suitability=True,
    embedding=True,  # IMPORTANT: enables /v1/embeddings
    pooling="mean"  # this fixes /v1/embeddings
)


Loading model: gemma-3-1b-Q4_K_M

Model: gemma-3-1b-Q4_K_M
Description: Gemma 3 1B instruct (Q4_K_M) - Good for 1GB VRAM
Size: 700 MB (~0.7 GB)
Minimum VRAM: 0.5 GB
Source: https://huggingface.co/unsloth/gemma-3-1b-it-GGUF
Cache location: /usr/local/lib/python3.12/dist-packages/llamatelemetry/models/gemma-3-1b-it-Q4_K_M.gguf



Download this model? [Y/n]:  y



This may take a while (700 MB)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:190: UserWarning: The `resume_download` argument is deprecated and ignored in `hf_hub_download`. Downloads always resume whenever possible.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


gemma-3-1b-it-Q4_K_M.gguf:   0%|          | 0.00/806M [00:00<?, ?B/s]

✓ Model downloaded: gemma-3-1b-it-Q4_K_M.gguf

GGUF Suitability Report:
  Model: Gemma-3-1B-It
  Size: 0.751 GB
  Quant: unknown
  Fits: True
  Recommended Quant: Q8_0
  Recommended GPU Layers: 99

Starting llama-server...
GPU Check:
  Platform: kaggle
  GPU: Tesla T4
  Compute Capability: 7.5
  Status: ✓ Compatible
  Command: /usr/local/lib/python3.12/dist-packages/llamatelemetry/binaries/cuda12/llama-server -m /usr/local/lib/python3.12/dist-packages/llamatelemetry/models/gemma-3-1b-it-Q4_K_M.gguf --host 127.0.0.1 --port 8080 -ngl 99 -c 4096 --parallel 4 -b 512 -ub 128 --metrics --props --embeddings --pooling mean
Starting llama-server...
  Executable: /usr/local/lib/python3.12/dist-packages/llamatelemetry/binaries/cuda12/llama-server
  Model: gemma-3-1b-it-Q4_K_M.gguf
  GPU Layers: 99
  Context Size: 4096
  Server URL: http://127.0.0.1:8080
Waiting for server to be ready...... ✓ Ready in 3.0s

✓ Model loaded and ready for inference
  Server: http://127.0.0.1:8080
  GPU Layers: 99
  C

True

In [7]:
# Run a single inference with sampling parameters.
result = engine.infer(
    prompt="Explain what CUDA cores are in two sentences.",
    max_tokens=128,      # maximum tokens to generate
    temperature=0.7,     # sampling temperature
    top_p=0.9,           # nucleus sampling threshold
    top_k=40,            # top-k sampling
    seed=0,              # 0 = random seed
    stop_sequences=None, # optional list of stop strings
)

{
    "name": "llm.inference",
    "context": {
        "trace_id": "0x7a483aca5871e3e7f1a955e8afb4464f",
        "span_id": "0x4c406dc548597f7e",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": null,
    "start_time": "2026-03-10T09:11:41.025496Z",
    "end_time": "2026-03-10T09:11:41.607522Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "llm.system": "llamatelemetry",
        "llm.model": "gemma-3-1b-it-Q4_K_M.gguf",
        "llm.input.tokens": 8,
        "llm.output.tokens": 48,
        "llm.latency_ms": 581.9270610809326,
        "llm.tokens_per_sec": 82.48456414939656,
        "gpu.device_id": 0,
        "nccl.split_mode": "none",
        "gen_ai.provider.name": "llama.cpp",
        "gen_ai.system": "llama.cpp",
        "gen_ai.request.model": "gemma-3-1b-it-Q4_K_M.gguf",
        "gen_ai.response.model": "gemma-3-1b-it-Q4_K_M.gguf",
        "gen_ai.usage.input_tokens": 8,
        "gen_ai.usage.output_tokens

In [8]:
# Print inference results and basic performance stats.
print(f"Success:          {result.success}")
print(f"Generated text:   {result.text}")
print(f"Tokens generated: {result.tokens_generated}")
print(f"Latency:          {result.latency_ms:.1f} ms")
print(f"Throughput:       {result.tokens_per_sec:.1f} tokens/sec")

if not result.success:
    print(f"Error: {result.error_message}")

Success:          True
Generated text:   

CUDA cores are specialized processing units within a GPU that allow for parallel computation, accelerating tasks like graphics rendering and scientific simulations.

They are implemented with a large number of tiny cores designed to work together to efficiently perform complex calculations.

Tokens generated: 48
Latency:          581.9 ms
Throughput:       82.5 tokens/sec


In [9]:
# Demonstrate deterministic, creative, and stop-sequence decoding.
# Deterministic output (greedy decoding)
result = engine.infer(
    "List three benefits of quantization.",
    max_tokens=256,
    temperature=0.0,
    top_k=1,
)

print(result)
print("=" * 100)

# Creative output
result = engine.infer(
    "Write a short poem about GPU computing.",
    max_tokens=200,
    temperature=1.2,
    top_p=0.95,
    top_k=100,
)

print(result)
print("=" * 100)

# Stop at specific sequences
result = engine.infer(
    "Q: What is GGUF?\nA:",
    max_tokens=128,
    stop_sequences=["\nQ:", "\n\n"],
)

print(result)



a)  Reduced memory usage
b)  Increased processing speed
c)  Improved accuracy
d)  All of the above

**Correct Answer: d) All of the above**

Here's why:

*   **a) Reduced memory usage:** Quantization reduces the precision of the numbers used to represent the data. Lower precision means less memory is needed to store the data.
*   **b) Increased processing speed:**  Computers can perform operations on lower-precision numbers more efficiently, leading to faster processing.
*   **c) Improved accuracy:**  Using lower precision can sometimes lead to a reduction in errors, especially when combined with other techniques like pruning.

Let me know if you'd like a more detailed explanation of any of these points!


The silicon sings, a digital art,
A GPU’s dance, a powerful start.
Layers unfold, with speed and grace,
Processing data at a rapid pace.

The pixels glow, a vibrant hue,
Through complex tasks, it sees it through.
Physics and vision, finely spun,
GPU computing, a victory won.

It’s 

In [10]:
# Run batch inference across multiple prompts.
prompts = [
  "Explain tensor cores in one sentence.",
  "What is the GGUF file format?",
  "How does KV cache work in transformers?",
  "Describe continuous batching for LLM serving.",
]

results = engine.batch_infer(prompts, max_tokens=96)

for i, r in enumerate(results):
  print(f"\n--- Prompt {i + 1} ---")
  print(f"Text: {r.text}")
  print(f"Tokens/sec: {r.tokens_per_sec:.1f}")



--- Prompt 1 ---
Text: 

Tensor cores are specialized processing units designed to accelerate matrix operations, like those used in deep learning, by performing computations on multiple data points simultaneously.

---

Here's a breakdown of the key concepts:

*   **Matrix Operations:** Deep learning models rely heavily on matrix multiplications, which are the core of most neural network calculations.
*   **Specialized Units:** Tensor cores are optimized for these operations, offering significant speed improvements.
*   **Parallel Processing:**
Tokens/sec: 94.8

--- Prompt 2 ---
Text: 

GGUF is a file format designed for running large language models (LLMs) on CPUs. It's a key part of the growing trend towards running these models on consumer-grade hardware.

Here's a breakdown of its key features and characteristics:

* **Format:** It's a binary format, meaning it's a structured file that contains the model's parameters, the necessary software libraries, and other configuration data.

In [11]:
# Create an embedding engine and compute a single embedding.
from llamatelemetry import InferenceEngine
from llamatelemetry.embeddings import EmbeddingEngine

embedder = EmbeddingEngine(engine)

embedding = embedder.embed("AI is transforming the world")
print(f"Embedding dimension: {len(embedding)}")
print(f"First 5 values: {embedding[:5]}")

Embedding dimension: 1152
First 5 values: [ 0.00517522  0.00697909  0.01545575 -0.01180242  0.02807245]


In [ ]:
# Placeholder: add follow-up setup or experiments here.


In [12]:
# Create a low-level API client for llama.cpp-compatible endpoints.
from llamatelemetry.api import LlamaCppClient
client = LlamaCppClient(base_url="http://127.0.0.1:8080")

In [14]:
# Run a chat completion request via the client.
response = client.chat_completion(
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is CUDA?"},
    ],
    max_tokens=200,
    temperature=0.7,
)
print(response.choices[0].message.content)

Okay, let's break down what CUDA is – it’s a really important and often misunderstood technology! Here's a breakdown in plain language:

**CUDA (Compute Unified Device Architecture) is a parallel computing platform and programming model developed by NVIDIA.**  Think of it as a way to get computers to do more calculations at the same time, dramatically speeding up complex tasks.

Here’s a deeper dive into what makes it special:

**1. What it Does:**

* **Leverages GPUs (Graphics Processing Units):** Traditionally, GPUs were designed primarily for graphics rendering (like games and videos). However, NVIDIA developed GPUs with *thousands* of cores – much more than CPUs (the "brains" of your computer).
* **Parallel Computing:** CUDA allows developers to write code that’s specifically tailored to run on these powerful GPUs.  Instead of a CPU doing everything sequentially, you can divide the problem into smaller pieces and have many cores working simultaneously.
*


In [15]:
# Run a standard completion request via the client.

response = client.complete(
  prompt="The three main benefits of model quantization are:",
  n_predict=128,
  temperature=0.5,
)

print(response.choices[0].text)




*   **Reduced Model Size:** Quantized models require less memory, making them easier to deploy on resource-constrained devices like mobile phones and embedded systems.
*   **Faster Inference Speed:**  Lower precision arithmetic (e.g., using 8-bit integers instead of 32-bit floating-point) can significantly speed up inference because computations are performed faster.
*   **Lower Energy Consumption:** Reduced precision leads to lower power consumption during inference, which is crucial for battery-powered devices.

Let me know if you'd like a more detailed explanation of any of these benefits!


In [16]:
# Request embeddings via the client and inspect the vector.
response = client.embeddings.create(
  input="GPU-accelerated inference with llamatelemetry",
)

embedding = response.data[0].embedding
print(f"Embedding dimension: {len(embedding)}")
print(f"First 5 values: {embedding[:5]}")

Embedding dimension: 1152
First 5 values: [0.029734354466199875, -0.00037454254925251007, 0.04345717653632164, -0.022549454122781754, 0.005893060937523842]


In [17]:
# Tokenize and detokenize text using the client.
tokens = client.tokenize("Hello, llamatelemetry!")

print(f"Token count: {len(tokens.tokens)}")
print(f"Token IDs: {tokens.tokens}")

text = client.detokenize(tokens.tokens)
print(f"Detokenized: {text}")

Token count: 6
Token IDs: [9259, 236764, 20871, 79737, 54539, 236888]
Detokenized: Hello, llamatelemetry!


In [18]:
# Check server health status via the client.
health = client.health()
print(f"Server status: {health}")


Server status: HealthStatus(status='ok', slots_idle=None, slots_processing=None)


In [19]:
# Stream chat completion tokens and print incrementally.
# Streaming chat completions
for chunk in client.chat.completions.create(
  messages=[{"role": "user", "content": "Explain flash attention step by step."}],
  max_tokens=256,
  stream=True,
):
  delta = chunk["choices"][0].get("delta", {})
  content = delta.get("content", "")
  print(content, end="", flush=True)

print()

NoneOkay, let's break down FlashAttention – a revolutionary technique designed to speed up training large Transformer models like GPT-3 and Llama 2. It tackles a crucial bottleneck in these models: the memory efficiency of matrix multiplications during forward and backward passes. Here’s a step-by-step explanation:

**1. The Problem: Matrix Multiplication Bottleneck**

* **Transformers rely heavily on matrix multiplication.** These matrices are fundamental to understanding the relationships between words in a sentence. During training, these operations happen billions of times per second.
* **Traditional attention mechanisms (like scaled dot-product attention) can be extremely slow when dealing with huge sequences.**  The standard implementation often involves multiplying large matrices repeatedly using traditional libraries like NumPy or PyTorch.
* **This process requires massive amounts of memory.** The data needs to be stored in RAM, and the computation itself is very expensive. Thi

In [20]:
# Fetch engine metrics snapshot.
metrics = engine.get_metrics()
print(metrics)

{'latency': {'mean_ms': 826.2050747871399, 'p50_ms': 758.44407081604, 'p95_ms': 1552.1714687347412, 'p99_ms': 1552.1714687347412, 'min_ms': 250.03600120544434, 'max_ms': 1552.1714687347412, 'sample_count': 8}, 'throughput': {'total_tokens': 758, 'total_requests': 8, 'tokens_per_sec': 114.68097073164432, 'requests_per_sec': 1.2103532531044257}}


In [21]:
# Configure telemetry and run a sample inference.
engine = lt.InferenceEngine(
    enable_telemetry=True,
    telemetry_config={
        "service_name": "my-llm-service",
        "enable_llama_metrics": True,
    },
)


result = engine.infer("What is OpenTelemetry?", max_tokens=64)

# Traces and metrics are now being collected
# Configure OTLP export via environment variables:
#   OTEL_EXPORTER_OTLP_ENDPOINT=https://your-collector:4318
#   OTEL_EXPORTER_OTLP_HEADERS=Authorization=Basic ...

{
    "name": "llm.inference",
    "context": {
        "trace_id": "0x618e795f64da585d5e443b53bbb93b0a",
        "span_id": "0xdec777fe7d444ba8",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": null,
    "start_time": "2026-03-10T09:14:13.706033Z",
    "end_time": "2026-03-10T09:14:13.712827Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "llm.error": "Connection error: HTTPConnectionPool(host='127.0.0.1', port=8090): Max retries exceeded with url: /completion (Caused by NewConnectionError(\"HTTPConnection(host='127.0.0.1', port=8090): Failed to establish a new connection: [Errno 111] Connection refused\"))"
    },
    "events": [
        {
            "name": "exception",
            "timestamp": "2026-03-10T09:14:13.712800Z",
            "attributes": {
                "exception.type": "requests.exceptions.ConnectionError",
                "exception.message": "HTTPConnectionPool(host='127.0.0.1', port=8090):

In [ ]:
# Placeholder: add follow-up setup or experiments here.


In [ ]:
# Placeholder: add follow-up setup or experiments here.


In [ ]:
# Placeholder: add follow-up setup or experiments here.


In [ ]:
# Placeholder: add follow-up setup or experiments here.
